# CHSL Coach: one-click fine-tune -> GGUF for GGUF Lab

**How to run**
1. **Runtime -> Change runtime type -> T4 GPU -> Save**
2. **Runtime -> Run all**
3. Allow Google Drive access when the pop-up appears (cell 1). Then you can leave it running.

Total time: about **45-75 minutes**. When it finishes, your model is in Google Drive at **`MyDrive/chsl_coach/chsl-coach-Q4_K_M.gguf`** (~1 GB).
Copy it to your phone and open it in GGUF Lab with **Load model**.

Everything is inside this notebook. You don't need to upload anything.

## 1. Connect Google Drive (the finished model is saved here)

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/chsl_coach"
os.makedirs(SAVE_DIR, exist_ok=True)
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
      or "NO GPU! Go to Runtime -> Change runtime type -> T4 GPU, then Run all again.")

## 2. Settings (change only if you want)

In [ ]:
BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"   # phone size. Try "unsloth/Qwen2.5-3B-Instruct" for a stronger, 2x bigger model
N_QUESTIONS = 6000      # generated training questions
EPOCHS      = 2
LR          = 2e-4
LORA_RANK   = 32
MAX_LEN     = 1024
OUT_NAME    = "chsl-coach"
QUANT       = "q4_k_m"  # q4_k_m = best for phones; "q8_0" = higher quality, 2x size

## 3. Install Unsloth (3-5 min)

In [ ]:
%%capture
!pip install unsloth

## 4. Build the training data (verified answers, ~10 seconds)
To add your own previous-year questions, put a `my_questions.jsonl` file (same format as `data/my_questions_template.jsonl`) in `MyDrive/chsl_coach/` before running.

In [ ]:
import base64, io, tarfile, os, shutil, subprocess
os.chdir("/content")
KIT = """
...
"""
shutil.rmtree("chsl_tutor", ignore_errors=True)
tarfile.open(fileobj=io.BytesIO(base64.b64decode("".join(KIT.split()))), mode="r:gz").extractall(".")
os.chdir("/content/chsl_tutor")
cmd = ["python", "build_dataset.py", "--n", str(N_QUESTIONS)]
extra = os.path.join(SAVE_DIR, "my_questions.jsonl")
if os.path.exists(extra):
    cmd += ["--extra", extra]
    print("Using your own questions from", extra)
print(subprocess.run(cmd, capture_output=True, text=True, check=True).stdout)

## 5. Load the base model (4-bit) and add LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_LEN, load_in_4bit=True, dtype=None)
model = FastLanguageModel.get_peft_model(
    model, r=LORA_RANK, lora_alpha=LORA_RANK, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

## 6. Prepare the dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"})
def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}
ds = ds.map(to_text, batched=True, remove_columns=ds["train"].column_names)
print(ds)
print(ds["train"][0]["text"][:800])

## 7. Train (30-50 min on a T4). Loss is computed only on the coach's replies.

In [ ]:
import inspect
from trl import SFTTrainer, SFTConfig

cfg = dict(
    dataset_text_field="text",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    optim="adamw_8bit",
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    per_device_eval_batch_size=8,
    save_strategy="no",
    seed=3407,
    output_dir="outputs",
    report_to="none",
)
sft_params = inspect.signature(SFTConfig.__init__).parameters
cfg["max_length" if "max_length" in sft_params else "max_seq_length"] = MAX_LEN   # name changed across TRL versions

trainer_kw = dict(model=model, train_dataset=ds["train"], eval_dataset=ds["validation"], args=SFTConfig(**cfg))
trainer_params = inspect.signature(SFTTrainer.__init__).parameters
trainer_kw["processing_class" if "processing_class" in trainer_params else "tokenizer"] = tokenizer
trainer = SFTTrainer(**trainer_kw)
trainer = train_on_responses_only(trainer,
                                  instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(stats)
model.save_pretrained(f"{SAVE_DIR}/{OUT_NAME}-lora")      # backup of the adapter in Drive
tokenizer.save_pretrained(f"{SAVE_DIR}/{OUT_NAME}-lora")

## 8. Try a few questions

In [ ]:
FastLanguageModel.for_inference(model)
SYSTEM = open("system_prompt.txt", encoding="utf-8").read().strip()

def ask(q, max_new_tokens=256):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    enc = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    ids = enc["input_ids"].to("cuda")
    out = model.generate(input_ids=ids, attention_mask=enc["attention_mask"].to("cuda"),
                         max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

for q in [
    "The price of sugar rises by 20%. By how much percent must a family cut consumption to keep expenditure the same?",
    "A can do a work in 10 days and B in 15 days. In how many days will they finish it together?",
    "Find the angle between the hands of a clock at 4:20.",
    "Spot the error: Each of the students have submitted the form.",
    "Who is the current RBI governor?",
]:
    print("Q:", q); print(ask(q)); print("-" * 60)

## 9. Score on 150 held-out questions (never seen in training)

In [ ]:
import json, re
evals = [json.loads(l) for l in open("data/eval.jsonl", encoding="utf-8")]
def norm(s): return re.sub(r"\s+", " ", s.strip().lower())
def is_correct(reply, gold):
    m = re.search(r"Answer:\s*(.*)", reply)
    if not m: return False
    line = norm(m.group(1))
    return line.startswith(f"({gold['letter']})") if gold["letter"] else norm(gold["answer"]) in line

N = 150
hits, by_fam = 0, {}
for g in evals[:N]:
    ok = is_correct(ask(g["question"], 200), g)
    hits += ok
    f = by_fam.setdefault(g["family"], [0, 0]); f[0] += ok; f[1] += 1
report = f"Accuracy: {hits}/{N} = {hits/N:.1%}\n" + "\n".join(
    f"{fam:34s} {a}/{b}" for fam, (a, b) in sorted(by_fam.items(), key=lambda x: x[1][0] / x[1][1]))
print(report)
open(f"{SAVE_DIR}/{OUT_NAME}-eval.txt", "w").write(report)

## 10. Export GGUF and save to Drive (5-15 min)

In [ ]:
import glob, os, shutil, subprocess
os.chdir("/content")
final = f"{SAVE_DIR}/{OUT_NAME}-{QUANT.upper()}.gguf"

def find_gguf():
    files = [f for f in glob.glob("/content/**/*.gguf", recursive=True) if QUANT.upper() in os.path.basename(f).upper()]
    return max(files, key=os.path.getmtime) if files else None

try:
    model.save_pretrained_gguf(OUT_NAME, tokenizer, quantization_method=QUANT)
except Exception as e:
    print("Unsloth GGUF export failed, using llama.cpp directly:", e)

src = find_gguf()
if src is None:   # fallback: merge -> convert -> quantize with llama.cpp
    model.save_pretrained_merged(f"{OUT_NAME}-merged", tokenizer, save_method="merged_16bit")
    if not os.path.exists("llama.cpp"):
        subprocess.run("git clone --depth 1 https://github.com/ggml-org/llama.cpp", shell=True, check=True)
    subprocess.run("pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt", shell=True)
    subprocess.run(f"python llama.cpp/convert_hf_to_gguf.py {OUT_NAME}-merged --outtype f16 --outfile {OUT_NAME}-f16.gguf", shell=True, check=True)
    subprocess.run("cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF && cmake --build llama.cpp/build --target llama-quantize -j", shell=True, check=True)
    q = glob.glob("llama.cpp/build/bin/llama-quantize")[0]
    subprocess.run(f"{q} {OUT_NAME}-f16.gguf {OUT_NAME}-{QUANT.upper()}.gguf {QUANT.upper()}", shell=True, check=True)
    src = find_gguf()

shutil.copy(src, final)
print(f"DONE -> {final}  ({os.path.getsize(final) / 1e9:.2f} GB)")
print("Copy it to your phone and open it in GGUF Lab with 'Load model'.")